# Part 1: Data collection and preparation

Owner: _(name)_

Brief, section 1: record the source URL, series identifier, access date, period and sample size and
justify the period; check chronological order, duplicates, missing observations and outliers;
distinguish market closures from missing data; explain any cleaning, scaling or transformation.

This notebook **reports**. It never changes `data/sp500_daily.csv` without a group decision.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
y = common.target_series(df)
train, val, test = common.split(y)
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train {train.index.min().date()} to {train.index.max().date()} ({len(train)}) | validation {val.index.min().date()} to {val.index.max().date()} ({len(val)}) | test {test.index.min().date()} to {test.index.max().date()} ({len(test)})")
df.tail(3)

## 1. Source record (written by `data/download_sp500.py`)

In [ ]:
import json
meta = json.loads((root / "data" / "sp500_daily.meta.json").read_text())
for k, v in meta.items():
    print(f"{k:24s} {v}")

**Justify the period here** (why this start date: enough history for the models, recent enough to be
relevant, includes calm and stressed markets, ...).

## 2. First look

In [ ]:
df.info()
df[["Open", "High", "Low", "Close", "Volume"]].describe().round(2)

## 3. Chronological order, duplicates, missing values

In [ ]:
print("index sorted in time order:", df.index.is_monotonic_increasing)
print("duplicate dates:", int(df.index.duplicated().sum()))
print("missing values per column:"); print(df.isna().sum())

## 4. Market closures are not missing data

The index only exists on trading days. Weekends and exchange holidays are closures, not gaps. A real
gap is a trading day with no row. The check below compares the dates in the file with the NYSE
calendar (weekdays minus holidays); if the calendar package is not installed it falls back to
listing the longest gaps so you can inspect them by hand.

In [ ]:
try:
    import pandas_market_calendars as pmc
    nyse = pmc.get_calendar("NYSE")
    sched = nyse.schedule(start_date=df.index.min(), end_date=df.index.max())
    trading_days = pd.DatetimeIndex(sched.index.date)
    missing = trading_days.difference(df.index.normalize())
    extra = df.index.normalize().difference(trading_days)
    print("trading days in calendar:", len(trading_days), "| rows in file:", len(df))
    print("trading days with no row (real gaps):", len(missing)); print(missing[:20])
    print("rows on non-trading days:", len(extra)); print(extra[:20])
except ImportError:
    print("pandas_market_calendars not installed (pip install pandas_market_calendars); showing the longest gaps instead")
    gaps = df.index.to_series().diff().dt.days
    print(gaps.sort_values(ascending=False).head(10))

## 5. Outliers

For a price series an "outlier" is a day with an extreme return, not an extreme price. Large moves
such as March 2020 are real events, not errors: list them, explain them, keep them, and say so.

In [ ]:
r = df["log_return"].dropna()
z = (r - r.mean()) / r.std()
extreme = r[z.abs() > 4]
print(len(extreme), "days beyond 4 standard deviations")
print(extreme.sort_values().head(10)); print(extreme.sort_values().tail(10))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
df["Close"].plot(ax=ax[0], title="S&P 500 daily close (index points)")
r.plot(ax=ax[1], title="daily log return")
plt.tight_layout()

## 6. Cleaning, scaling and transformations: the decisions

One line each, with the reason, for example:

- No missing trading days, no duplicate dates: nothing to impute or drop.
- Extreme days kept: real market events the models must face.
- Transformation: the models work on the daily log return (`src/config.py`), which is stationary;
  every forecast is converted back to a closing value before scoring (`common.to_price`).
- Scaling: the neural networks scale inputs with a scaler fitted on the training set only.

## 7. The split (from `src/config.py`), chronological, no shuffling

In [ ]:
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:11s} {part.index.min().date()} to {part.index.max().date()}  {len(part):5d} days")